In [1]:
import os

BASE_DIR      = r"D:\FCAI\plain_coupon\infared_data"
OUTPUT_SUBDIR = "time_average"
SKIP_FOLDERS  = {"time_average", "time_averaged_plot_intensity"}

base      = os.path.normpath(BASE_DIR)
entries   = sorted(os.listdir(base))
case_dirs = [
    os.path.join(base, e) for e in entries
    if os.path.isdir(os.path.join(base, e)) and e not in SKIP_FOLDERS
]

for case_dir in case_dirs:
    case_name = os.path.basename(case_dir)
    f1 = os.path.join(case_dir, OUTPUT_SUBDIR, "window_1_corrected.csv")
    f2 = os.path.join(case_dir, OUTPUT_SUBDIR, "window_2_corrected.csv")
    print(f"[{case_name}]")
    print(f"  window_1_corrected.csv : {'FOUND' if os.path.isfile(f1) else 'MISSING'}")
    print(f"  window_2_corrected.csv : {'FOUND' if os.path.isfile(f2) else 'MISSING'}")

[.ipynb_checkpoints]
  window_1_corrected.csv : MISSING
  window_2_corrected.csv : MISSING
[anaconda_projects]
  window_1_corrected.csv : MISSING
  window_2_corrected.csv : MISSING
[cr_1400_phi_085]
  window_1_corrected.csv : FOUND
  window_2_corrected.csv : FOUND
[cr_1400_phi_09]
  window_1_corrected.csv : FOUND
  window_2_corrected.csv : FOUND
[cr_1400_phi_10]
  window_1_corrected.csv : FOUND
  window_2_corrected.csv : FOUND
[cr_1400_phi_118]
  window_1_corrected.csv : FOUND
  window_2_corrected.csv : FOUND
[cr_1400_phi_137]
  window_1_corrected.csv : FOUND
  window_2_corrected.csv : FOUND
[cr_2300_phi_085]
  window_1_corrected.csv : FOUND
  window_2_corrected.csv : FOUND
[cr_2300_phi_09]
  window_1_corrected.csv : FOUND
  window_2_corrected.csv : FOUND
[cr_2300_phi_10]
  window_1_corrected.csv : FOUND
  window_2_corrected.csv : FOUND
[cr_2300_phi_118]
  window_1_corrected.csv : FOUND
  window_2_corrected.csv : FOUND
[cr_2300_phi_137]
  window_1_corrected.csv : FOUND
  window_2_corre

In [2]:
"""
Paste this entire cell into your Jupyter notebook and run it.

For every case folder under BASE_DIR, reads:
    time_average/window_1_corrected.csv
    time_average/window_2_corrected.csv

Interpolates each to 150 x 150 using 2D cubic interpolation, and saves:
    time_average/window_1_corrected_interp.csv
    time_average/window_2_corrected_interp.csv

The output keeps the same 9-line header format as the input.
"""

import os
import numpy as np
from scipy.interpolate import RectBivariateSpline

# ── Configuration ─────────────────────────────────────────────────────────────
BASE_DIR       = r"D:\FCAI\plain_coupon\infared_data"
OUTPUT_SUBDIR  = "time_average"
HEADER_LINES   = 9
TARGET_ROWS    = 150
TARGET_COLS    = 150
SKIP_FOLDERS   = {"time_average", "time_averaged_plot_intensity"}

# Files to interpolate and their output names
FILES_TO_INTERPOLATE = {
    "window_1_corrected.csv": "window_1_corrected_interp.csv",
    "window_2_corrected.csv": "window_2_corrected_interp.csv",
}
# ──────────────────────────────────────────────────────────────────────────────


def read_csv(filepath):
    """Return (header_lines, 2D numpy array)."""
    with open(filepath, "r") as f:
        lines = f.readlines()
    header     = lines[:HEADER_LINES]
    data_lines = [l.strip() for l in lines[HEADER_LINES + 1:] if l.strip()]
    data = np.array(
        [[float(v) for v in row.split(",")] for row in data_lines],
        dtype=np.float64,
    )
    return header, data


def write_csv(out_path, header_lines, data, note):
    """Write header + data in scientific notation, updating the header note."""
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        for line in header_lines:
            stripped = line.rstrip("\n").rstrip("\r")
            if stripped.startswith("Filename"):
                f.write(f"{stripped}  [{note}]\n")
            else:
                f.write(line if line.endswith("\n") else line + "\n")
        f.write("\n")
        for row in data:
            f.write(",".join(f"{v:.6e}" for v in row) + "\n")


def interpolate_2d(data, target_rows, target_cols):
    """
    Resample a 2D array to (target_rows x target_cols) using
    bicubic spline interpolation (RectBivariateSpline, k=3).

    Normalised coordinates [0, 1] are used so the output spans
    the same spatial extent as the input regardless of size.
    """
    src_rows, src_cols = data.shape

    # Source coordinate axes (normalised 0→1)
    row_src = np.linspace(0, 1, src_rows)
    col_src = np.linspace(0, 1, src_cols)

    # Target coordinate axes (normalised 0→1)
    row_tgt = np.linspace(0, 1, target_rows)
    col_tgt = np.linspace(0, 1, target_cols)

    spline       = RectBivariateSpline(row_src, col_src, data, kx=3, ky=3)
    interpolated = spline(row_tgt, col_tgt)

    return interpolated


# ── Main ──────────────────────────────────────────────────────────────────────

base    = os.path.normpath(BASE_DIR)
entries = sorted(os.listdir(base))
case_dirs = [
    os.path.join(base, e) for e in entries
    if os.path.isdir(os.path.join(base, e)) and e not in SKIP_FOLDERS
]

print("=" * 66)
print(f"  2D Interpolation  →  {TARGET_ROWS} × {TARGET_COLS}")
print("=" * 66)
print(f"  Found {len(case_dirs)} case folder(s).\n")

total_ok   = 0
total_skip = 0

for case_dir in case_dirs:
    case_name = os.path.basename(case_dir)
    print(f"[{case_name}]")

    for src_name, dst_name in FILES_TO_INTERPOLATE.items():
        src_path = os.path.join(case_dir, OUTPUT_SUBDIR, src_name)
        dst_path = os.path.join(case_dir, OUTPUT_SUBDIR, dst_name)

        if not os.path.isfile(src_path):
            print(f"  [SKIP] '{src_name}' not found")
            total_skip += 1
            continue

        header, data = read_csv(src_path)
        src_rows, src_cols = data.shape

        interpolated = interpolate_2d(data, TARGET_ROWS, TARGET_COLS)

        note = f"INTERPOLATED {src_rows}x{src_cols} -> {TARGET_ROWS}x{TARGET_COLS}"
        write_csv(dst_path, header, interpolated, note)

        print(f"  [OK] {src_name}  ({src_rows} x {src_cols})"
              f"  ->  {dst_name}  ({TARGET_ROWS} x {TARGET_COLS})")
        total_ok += 1

print("\n" + "=" * 66)
print("  SUMMARY")
print("=" * 66)
print(f"  Files interpolated : {total_ok}")
print(f"  Files skipped      : {total_skip}")
print("=" * 66)
print(f"\n  Output saved as  '{list(FILES_TO_INTERPOLATE.values())[0]}'")
print(f"  and  '{list(FILES_TO_INTERPOLATE.values())[1]}'")
print(f"  under each case's  {OUTPUT_SUBDIR}/  folder.")

  2D Interpolation  →  150 × 150
  Found 17 case folder(s).

[.ipynb_checkpoints]
  [SKIP] 'window_1_corrected.csv' not found
  [SKIP] 'window_2_corrected.csv' not found
[anaconda_projects]
  [SKIP] 'window_1_corrected.csv' not found
  [SKIP] 'window_2_corrected.csv' not found
[cr_1400_phi_085]
  [OK] window_1_corrected.csv  (112 x 110)  ->  window_1_corrected_interp.csv  (150 x 150)
  [OK] window_2_corrected.csv  (114 x 114)  ->  window_2_corrected_interp.csv  (150 x 150)
[cr_1400_phi_09]
  [OK] window_1_corrected.csv  (112 x 110)  ->  window_1_corrected_interp.csv  (150 x 150)
  [OK] window_2_corrected.csv  (114 x 114)  ->  window_2_corrected_interp.csv  (150 x 150)
[cr_1400_phi_10]
  [OK] window_1_corrected.csv  (112 x 110)  ->  window_1_corrected_interp.csv  (150 x 150)
  [OK] window_2_corrected.csv  (114 x 114)  ->  window_2_corrected_interp.csv  (150 x 150)
[cr_1400_phi_118]
  [OK] window_1_corrected.csv  (112 x 110)  ->  window_1_corrected_interp.csv  (150 x 150)
  [OK] window_2